
# Household Energy Consumption Forecasting
## Objective

Forecast short-term household electricity consumption using historical power usage data.
The project compares three forecasting approaches:
- ARIMA (Statistical Time Series)
- Prophet (Trend + Seasonality)
- XGBoost (Machine Learning)

## Evaluation metrics:
- MAE
- RMSE

## Setup and Library Imports

We import standard packages for mathematical operations (`numpy`), data manipulation (`pandas`), visualization (`matplotlib`, `seaborn`), evaluation metrics (`scikit-learn`), and time series forecasting models (`statsmodels` for ARIMA, `prophet`, and `xgboost`). This ensures our environment contains all the analytical tools needed for the task.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split

from xgboost import XGBRegressor

from prophet import Prophet

from statsmodels.tsa.arima.model import ARIMA

## Load the Dataset

We read the raw household energy consumption text file using `pandas`. We specify `;` as the separator and interpret `?` as missing values (`NaN`). Setting `low_memory=False` prevents warnings regarding mixed data types in columns during the initial reading of this large dataset (containing over 2 million rows).

In [2]:
df = pd.read_csv(
    r"C:\Users\Waleed Qamar\Downloads\energy-forecasting\data\household_power_consumption.txt",
    sep=";",
    na_values="?",
    low_memory=False
)

## Initial Data Profiling and Exploratory Overview

Before performing data cleaning, it is critical to profile the raw dataset. We will inspect its shape, check how the columns are typed, inspect statistics for numerical columns, and quantify missing values and duplicate rows. This initial audit informs the data cleaning strategy.

### Previewing the Data Structure

We display the first five rows of the dataset using `df.head()`. This lets us check column names, verify the format of raw date and time strings, and inspect the values of active power, reactive power, voltage, intensity, and sub-metering sensors.

In [3]:
df.head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


### Inspecting Column Data Types and Counts

We invoke `df.info()` to verify how Pandas typed the columns, check the total memory footprint of the dataset, and confirm that datetime column strings need conversion while the numeric columns have been correctly parsed as float64 values.

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2075259 entries, 0 to 2075258
Data columns (total 9 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Date                   str    
 1   Time                   str    
 2   Global_active_power    float64
 3   Global_reactive_power  float64
 4   Voltage                float64
 5   Global_intensity       float64
 6   Sub_metering_1         float64
 7   Sub_metering_2         float64
 8   Sub_metering_3         float64
dtypes: float64(7), str(2)
memory usage: 142.5 MB


### Summarizing Statistical Distributions

Using `df.describe()`, we look at summary statistics (mean, standard deviation, min, max, percentiles) for the numerical columns. This shows the ranges of energy metrics (e.g. active power typically averages ~1.09 kW but ranges up to 11.1 kW), helping us spot any anomalies or scale issues.

In [5]:
df.describe()

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
count,2.049280e+06,2.049280e+06,2.049280e+06,2.049280e+06,2.049280e+06,2.049280e+06,2.049280e+06
mean,1.091615e+00,1.237145e-01,2.408399e+02,4.627759e+00,1.121923e+00,1.298520e+00,6.458447e+00
std,1.057294e+00,1.127220e-01,3.239987e+00,4.444396e+00,6.153031e+00,5.822026e+00,8.437154e+00
min,7.600000e-02,0.000000e+00,2.232000e+02,2.000000e-01,0.000000e+00,0.000000e+00,0.000000e+00
25%,3.080000e-01,4.800000e-02,2.389900e+02,1.400000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,6.020000e-01,1.000000e-01,2.410100e+02,2.600000e+00,0.000000e+00,0.000000e+00,1.000000e+00
75%,1.528000e+00,1.940000e-01,2.428900e+02,6.400000e+00,0.000000e+00,1.000000e+00,1.700000e+01
max,1.112200e+01,1.390000e+00,2.541500e+02,4.840000e+01,8.800000e+01,8.000000e+01,3.100000e+01


### Identifying and Quantifying Missing Values

We call `df.isnull().sum()` to get the exact count of missing values (`NaN`) in each column. Recognizing missing values helps us decide if we need to drop records or perform imputation before modeling. In this case, we have exactly 25,979 missing values in the power-related columns.

In [6]:
df.isnull().sum()

Date                         0
Time                         0
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64

### Checking for Duplicate Rows

We run `df.duplicated().sum()` to check if there are duplicate records. Duplicate rows can skew distributions and evaluation metrics if present. The output shows 0 duplicates, confirming we do not have duplicate entries.

In [7]:
df.duplicated().sum()

np.int64(0)